In [0]:
dbutils.widgets.text("catalog", "claims_dev")

In [0]:
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE ml.claim_features AS
WITH base AS (
  SELECT
    f.claim_id, f.workshop_id, f.cause, f.source, f.reported_ts, f.estimate_amount,
    unix_timestamp(f.reported_ts) AS ts,
    p.plan, p.vehicle_value,
    datediff(f.reported_ts, p.start_date) AS policy_age_days
  FROM gold.fact_claim f
  LEFT JOIN gold.dim_policy p ON f.policy_sk = p.policy_sk
)
SELECT
  claim_id,
  reported_ts,
  plan,
  cause,
  source = 'tpa' AS is_tpa,
  policy_age_days,
  CAST(estimate_amount AS DOUBLE) AS estimate_amount,
  CAST(estimate_amount / vehicle_value AS DOUBLE) AS estimate_to_value,
  CAST(estimate_amount / avg(estimate_amount) OVER (
    PARTITION BY cause ORDER BY ts RANGE BETWEEN 7776000 PRECEDING AND 1 PRECEDING
  ) AS DOUBLE) AS estimate_vs_cause_avg,
  CASE WHEN workshop_id IS NOT NULL THEN count(*) OVER w30 END AS workshop_claims_30d,
  CASE WHEN workshop_id IS NOT NULL
       THEN CAST(avg(CASE WHEN policy_age_days < 60 THEN 1.0 ELSE 0.0 END) OVER w30 AS DOUBLE)
  END AS workshop_new_policy_share_30d
FROM base
WINDOW w30 AS (PARTITION BY workshop_id ORDER BY ts RANGE BETWEEN 2592000 PRECEDING AND 1 PRECEDING)
""")

spark.sql("ALTER TABLE ml.claim_features ALTER COLUMN claim_id SET NOT NULL")
spark.sql("ALTER TABLE ml.claim_features ADD CONSTRAINT claim_features_pk PRIMARY KEY (claim_id)")

In [0]:
display(spark.sql("""
SELECT
  i.outcome,
  count(*) AS claims,
  round(avg(f.policy_age_days), 3) AS policy_age_days,
  round(avg(f.estimate_to_value), 3) AS estimate_to_value,
  round(avg(f.estimate_vs_cause_avg), 3) AS estimate_vs_cause_avg,
  round(avg(f.workshop_claims_30d), 3) AS workshop_claims_30d,
  round(avg(f.workshop_new_policy_share_30d), 3) AS workshop_new_policy_share_30d
FROM ml.claim_features f
JOIN bronze.investigations i ON f.claim_id = i.claim_id
GROUP BY i.outcome
"""))
display(spark.sql("SELECT count(*) AS n FROM ml.claim_features"))